# Hands-on — Preparing Relational Data with Pandas

This notebook accompanies **A3 — Preparing Data for a Relational Database with Pandas**.

The objective is to transform one small flat transactional dataset into several related DataFrames that match a relational database design.

We will follow this workflow:

```text
FLAT DATAFRAME
      ↓
REVIEW SMALL STRUCTURAL ISSUES
      ↓
IDENTIFY TABLE DEPENDENCIES
      ↓
CREATE PARENT DATAFRAMES
      ↓
CREATE / PRESERVE PKs
      ↓
PROPAGATE IDs WITH merge()
      ↓
CREATE CHILD DATAFRAMES
      ↓
CREATE BRIDGE DATAFRAME
      ↓
VALIDATE PKs / FKs / ROW COUNTS
      ↓
EXPORT ONE CSV PER TABLE
      ↓
LOAD INTO MYSQL
```

> **Important:** this is **not** a Data Cleaning exercise.  
> The fake dataset is intentionally almost clean. We will only fix a couple of small inconsistencies that would otherwise affect the relational structure.


## 1. Imports and reproducibility

The dataset is generated directly in Python, so no external CSV file is needed.

We use a fixed random seed to make the example reproducible.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

pd.set_option("display.max_columns", None)


## 2. Generate a small fake transactional dataset

Each row represents **one product inside one order**.

The dataset includes:

- customers;
- cities and countries;
- orders;
- products and categories;
- quantities and prices.

A few values contain small inconsistencies such as capitalization or trailing whitespace. These are included only because they affect entity identification.


In [ ]:
customers = [
    ("alice@example.com", "Alice", "Madrid", "Spain"),
    ("bob@example.com", "Bob", "Barcelona", "Spain"),
    ("carla@example.com", "Carla", "Lisbon", "Portugal"),
    ("diego@example.com", "Diego", "Madrid", "Spain"),
    ("emma@example.com", "Emma", "Porto", "Portugal"),
]

products = [
    ("BK-001", "Data Stories", "Books", 24.90),
    ("BK-002", "SQL for Humans", "Books", 31.50),
    ("EL-001", "Wireless Mouse", "Electronics", 19.99),
    ("EL-002", "USB-C Hub", "Electronics", 44.50),
    ("HM-001", "Coffee Mug", "Home", 11.90),
    ("HM-002", "Desk Lamp", "Home", 28.00),
]

orders = [
    ("ORD-001", "2026-08-01", "alice@example.com"),
    ("ORD-002", "2026-08-02", "bob@example.com"),
    ("ORD-003", "2026-08-03", "carla@example.com"),
    ("ORD-004", "2026-08-04", "alice@example.com"),
    ("ORD-005", "2026-08-05", "diego@example.com"),
    ("ORD-006", "2026-08-06", "emma@example.com"),
    ("ORD-007", "2026-08-07", "bob@example.com"),
    ("ORD-008", "2026-08-08", "carla@example.com"),
]

rows = []

for order_number, order_date, customer_email in orders:
    customer_name, city, country = next(
        (name, city, country)
        for email, name, city, country in customers
        if email == customer_email
    )

    n_products = int(rng.integers(1, 4))
    chosen = rng.choice(len(products), size=n_products, replace=False)

    for idx in chosen:
        product_sku, product_name, category, unit_price = products[idx]

        rows.append(
            {
                "order_number": order_number,
                "order_date": order_date,
                "customer_email": customer_email,
                "customer_name": customer_name,
                "city": city,
                "country": country,
                "product_sku": product_sku,
                "product_name": product_name,
                "category": category,
                "quantity": int(rng.integers(1, 5)),
                "unit_price": unit_price,
            }
        )

df = pd.DataFrame(rows)

# Two deliberately small inconsistencies that affect relational modelling.
df.loc[df.index[2], "category"] = df.loc[df.index[2], "category"].lower()
df.loc[df.index[7], "city"] = df.loc[df.index[7], "city"] + " "

df.head(10)


## 3. Inspect the source DataFrame

Before creating any relational table, inspect the dataset.

Questions to answer:

- What does one row represent?
- Which columns look like identifiers?
- Which columns describe repeated concepts?
- Which fields may become entities?
- Which fields belong to relationships?


In [ ]:
print("Shape:", df.shape)
print()

print("Columns:")
print(df.columns.tolist())
print()

df.info()


In [ ]:
df.head(12)


### Quick check

In this dataset, one row represents:

> **one product included in one order**

That means the combination of `order_number` and `product_sku` is especially important.

The value `quantity` describes the relationship between an order and a product. It does **not** belong to the product or order independently.


## 4. Correct only the small inconsistencies that affect the relational structure

If we run `drop_duplicates()` before correcting inconsistent labels, values such as:

```text
Books
books
```

would become two different categories.

Similarly:

```text
Madrid
Madrid 
```

would become two different cities.

We will fix only these small issues.


In [ ]:
df["category"] = (
    df["category"]
    .str.strip()
    .str.title()
)

df["city"] = (
    df["city"]
    .str.strip()
    .str.title()
)

df[["category", "city"]].drop_duplicates().sort_values(
    ["category", "city"]
)


## 5. Relational model used in this notebook

We will prepare data for the following simplified ERD:

```text
COUNTRY
   1
   |
   N
 CITY
   1
   |
   N
CUSTOMER
   1
   |
   N
 ORDER
   1
   |
   N
ORDER_PRODUCT
   N
   |
   1
 PRODUCT
   N
   |
   1
CATEGORY
```

The bridge table `ORDER_PRODUCT` contains the relationship attributes:

```text
quantity
unit_price
```

The dependency order therefore begins with tables that do not contain foreign keys.


## 6. Create `COUNTRY`

`COUNTRY` does not depend on any other table.

We create a dedicated `country_id`.


In [ ]:
countries_df = (
    df[["country"]]
    .drop_duplicates()
    .sort_values("country")
    .reset_index(drop=True)
)

countries_df["country_id"] = range(1, len(countries_df) + 1)

countries_df = countries_df[
    ["country_id", "country"]
]

countries_df


### Validate the primary key


In [ ]:
assert countries_df["country_id"].is_unique
assert countries_df["country_id"].notna().all()

countries_df


## 7. Create `CITY`

`CITY` depends on `COUNTRY`.

We first propagate `country_id` using `merge()`.


In [ ]:
cities_df = (
    df[["city", "country"]]
    .drop_duplicates()
    .merge(
        countries_df,
        on="country",
        how="left",
        validate="many_to_one"
    )
)

cities_df = cities_df[
    ["city", "country_id"]
].drop_duplicates()

cities_df = cities_df.sort_values(
    ["country_id", "city"]
).reset_index(drop=True)

cities_df["city_id"] = range(1, len(cities_df) + 1)

cities_df = cities_df[
    ["city_id", "city", "country_id"]
]

cities_df


### Validate the merge

The `validate="many_to_one"` argument is useful because it asks Pandas to verify that many source rows can match only one parent row.

We also check that no city lost its foreign key.


In [ ]:
assert cities_df["city_id"].is_unique
assert cities_df["country_id"].notna().all()

cities_df


## 8. Create `CUSTOMER`

`CUSTOMER` depends on `CITY`.

The source dataset contains the city name, but the relational table needs `city_id`.


In [ ]:
customers_df = (
    df[
        [
            "customer_email",
            "customer_name",
            "city",
            "country"
        ]
    ]
    .drop_duplicates()
    .merge(
        countries_df,
        on="country",
        how="left",
        validate="many_to_one"
    )
    .merge(
        cities_df,
        on=["city", "country_id"],
        how="left",
        validate="many_to_one"
    )
)

customers_df = customers_df[
    [
        "customer_email",
        "customer_name",
        "city_id"
    ]
].drop_duplicates()

customers_df = customers_df.sort_values(
    "customer_email"
).reset_index(drop=True)

customers_df["customer_id"] = range(
    1,
    len(customers_df) + 1
)

customers_df = customers_df[
    [
        "customer_id",
        "customer_email",
        "customer_name",
        "city_id"
    ]
]

customers_df


In [ ]:
assert customers_df["customer_id"].is_unique
assert customers_df["customer_email"].is_unique
assert customers_df["city_id"].notna().all()


## 9. Create `CATEGORY`

`CATEGORY` is another independent parent table.


In [ ]:
categories_df = (
    df[["category"]]
    .drop_duplicates()
    .sort_values("category")
    .reset_index(drop=True)
)

categories_df["category_id"] = range(
    1,
    len(categories_df) + 1
)

categories_df = categories_df[
    ["category_id", "category"]
]

categories_df


In [ ]:
assert categories_df["category_id"].is_unique
assert categories_df["category"].is_unique


## 10. Create `PRODUCT`

`PRODUCT` depends on `CATEGORY`.

The original source already contains a stable identifier: `product_sku`.

We can preserve it as a business identifier while still creating our own dedicated `product_id`.


In [ ]:
products_df = (
    df[
        [
            "product_sku",
            "product_name",
            "category"
        ]
    ]
    .drop_duplicates()
    .merge(
        categories_df,
        on="category",
        how="left",
        validate="many_to_one"
    )
)

products_df = products_df[
    [
        "product_sku",
        "product_name",
        "category_id"
    ]
].drop_duplicates()

products_df = products_df.sort_values(
    "product_sku"
).reset_index(drop=True)

products_df["product_id"] = range(
    1,
    len(products_df) + 1
)

products_df = products_df[
    [
        "product_id",
        "product_sku",
        "product_name",
        "category_id"
    ]
]

products_df


In [ ]:
assert products_df["product_id"].is_unique
assert products_df["product_sku"].is_unique
assert products_df["category_id"].notna().all()


## 11. Create `ORDER`

`ORDER` depends on `CUSTOMER`.

We use `customer_email` as the temporary merge key because it identifies each customer in this fake dataset.


In [ ]:
orders_df = (
    df[
        [
            "order_number",
            "order_date",
            "customer_email"
        ]
    ]
    .drop_duplicates()
    .merge(
        customers_df[
            [
                "customer_id",
                "customer_email"
            ]
        ],
        on="customer_email",
        how="left",
        validate="many_to_one"
    )
)

orders_df["order_date"] = pd.to_datetime(
    orders_df["order_date"]
)

orders_df = orders_df[
    [
        "order_number",
        "order_date",
        "customer_id"
    ]
]

orders_df = orders_df.sort_values(
    "order_number"
).reset_index(drop=True)

orders_df["order_id"] = range(
    1,
    len(orders_df) + 1
)

orders_df = orders_df[
    [
        "order_id",
        "order_number",
        "order_date",
        "customer_id"
    ]
]

orders_df


In [ ]:
assert orders_df["order_id"].is_unique
assert orders_df["order_number"].is_unique
assert orders_df["customer_id"].notna().all()


## 12. Create the bridge table `ORDER_PRODUCT`

This is where the N:N relationship is resolved.

We need:

- `order_id`;
- `product_id`;
- `quantity`;
- `unit_price`.

Both IDs must be recovered from their parent DataFrames.


In [ ]:
order_product_df = (
    df[
        [
            "order_number",
            "product_sku",
            "quantity",
            "unit_price"
        ]
    ]
    .merge(
        orders_df[
            [
                "order_id",
                "order_number"
            ]
        ],
        on="order_number",
        how="left",
        validate="many_to_one"
    )
    .merge(
        products_df[
            [
                "product_id",
                "product_sku"
            ]
        ],
        on="product_sku",
        how="left",
        validate="many_to_one"
    )
)

order_product_df = order_product_df[
    [
        "order_id",
        "product_id",
        "quantity",
        "unit_price"
    ]
].reset_index(drop=True)

order_product_df["order_product_id"] = range(
    1,
    len(order_product_df) + 1
)

order_product_df = order_product_df[
    [
        "order_product_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price"
    ]
]

order_product_df.head(10)


In [ ]:
assert order_product_df["order_product_id"].is_unique
assert order_product_df["order_id"].notna().all()
assert order_product_df["product_id"].notna().all()


## 13. Final relational DataFrames

We now have one DataFrame per relational table.


In [ ]:
tables = {
    "countries": countries_df,
    "cities": cities_df,
    "customers": customers_df,
    "categories": categories_df,
    "products": products_df,
    "orders": orders_df,
    "order_product": order_product_df,
}

for name, table in tables.items():
    print(f"{name:15s} -> {table.shape}")


## 14. Check referential integrity in Pandas

Before exporting anything, verify that every foreign key points to an existing parent key.


In [ ]:
assert set(cities_df["country_id"]).issubset(
    set(countries_df["country_id"])
)

assert set(customers_df["city_id"]).issubset(
    set(cities_df["city_id"])
)

assert set(products_df["category_id"]).issubset(
    set(categories_df["category_id"])
)

assert set(orders_df["customer_id"]).issubset(
    set(customers_df["customer_id"])
)

assert set(order_product_df["order_id"]).issubset(
    set(orders_df["order_id"])
)

assert set(order_product_df["product_id"]).issubset(
    set(products_df["product_id"])
)

print("All foreign-key checks passed.")


## 15. Compare row counts with the source data

The bridge table should contain one row per source row because the original unit of observation was one product inside one order.


In [ ]:
print("Original rows:", len(df))
print("Bridge rows:  ", len(order_product_df))

assert len(df) == len(order_product_df)


## 16. Export one CSV per table

For this project, use:

```python
index=False
sep=";"
encoding="utf-8-sig"
```

The files will be stored in `data/clean/`.


In [ ]:
from pathlib import Path

output_dir = Path("data/clean")
output_dir.mkdir(parents=True, exist_ok=True)

for name, table in tables.items():
    table.to_csv(
        output_dir / f"{name}.csv",
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )

sorted(output_dir.glob("*.csv"))


### Optional validation: read one file back

If a CSV was exported with `sep=";"`, read it back using the same delimiter.


In [ ]:
check_df = pd.read_csv(
    output_dir / "products.csv",
    sep=";"
)

check_df.head()


## 17. Optional pattern — splitting a large DataFrame into chunks

The current dataset is far too small to require chunking.

The following pattern is included only because project datasets may be much larger.

Do **not** use chunking unless it solves a real loading or troubleshooting problem.


In [ ]:
# Example only — not necessary for this small dataset.

chunk_size = 100_000

# Uncomment for a genuinely large DataFrame:
#
# for part, start in enumerate(
#     range(0, len(order_product_df), chunk_size),
#     start=1
# ):
#     chunk = order_product_df.iloc[
#         start:start + chunk_size
#     ]
#
#     chunk.to_csv(
#         output_dir / f"order_product_part_{part}.csv",
#         index=False,
#         sep=";",
#         encoding="utf-8-sig"
#     )


## 18. Loading the CSV files with SQL

One possible MySQL approach is `LOAD DATA LOCAL INFILE`.

For example:

```sql
LOAD DATA LOCAL INFILE '/absolute/path/to/categories.csv'
INTO TABLE categories
FIELDS TERMINATED BY ';'
ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 ROWS
(category_id, category);
```

The exact path and configuration depend on your operating system and MySQL setup.

If the import produces warnings:

```sql
SHOW WARNINGS;
```

Typical causes of failure include:

- incompatible data types;
- missing required values;
- malformed rows;
- unexpected characters;
- delimiter or encoding problems;
- foreign-key violations;
- loading tables in the wrong dependency order.


## 19. Loading with MySQL Workbench

You can also use the **Table Data Import Wizard**.

Before importing:

```text
□ Select the correct schema.
□ Select the correct destination table.
□ Confirm that the delimiter is `;`.
□ Check the column mapping.
□ Confirm that the CSV columns match the MySQL table.
```

During the import:

> **Use `Show Logs` to inspect progress and diagnose failures.**

The logs can help identify the row or value that caused the import to stop.


## 20. Load in dependency order

For this example, a valid order is:

```text
1. countries
2. categories
3. cities
4. products
5. customers
6. orders
7. order_product
```

Other valid orders may exist as long as every parent row is loaded before a child row that references it.


## 21. Validate MySQL after loading

A successful import message is not enough.

Compare MySQL row counts with the Pandas DataFrames.

For example:

```sql
SELECT COUNT(*)
FROM categories;
```

and compare that result with:

```python
len(categories_df)
```

Inspect some rows:

```sql
SELECT *
FROM categories
LIMIT 10;
```

The final consistency check is:

```text
ERD
 ↕
PANDAS DATAFRAMES
 ↕
CSV FILES
 ↕
MYSQL TABLES
```


## 22. Final takeaway

The most reusable pattern from this notebook is:

```text
CREATE PARENT DATAFRAME
        ↓
CREATE / PRESERVE PK
        ↓
MERGE PK INTO DEPENDENT DATA
        ↓
CREATE CHILD DATAFRAME
        ↓
VALIDATE
        ↓
REPEAT
```

The difficult part is not `to_csv()`.

The difficult part is preserving the relationships while transforming one flat dataset into several related tables.
